# Experiment: one-hot CatBoost and class-weighted MLP

First on-site iteration: one-hot encodes `status`/`request_method`, CatBoost with a stratified hold-out, then a class-weighted Keras MLP.

In [ ]:
DATA_DIR = "../data"

In [128]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score

import catboost as cb
import lightgbm as lgb
import xgboost as xgb

import tensorflow as tf
from tensorflow.keras import backend as K

## Load Data

In [140]:
ori_train_df = pd.read_csv(f"{DATA_DIR}/train.csv", index_col=0)
ori_train_df.head()

,timestamp,remote_addr,remote_user,hostname,time_local,request,status,body_bytes_sent,http_referer,http_user_agent,...,upstream_connect_time,upstream_header_time,upstream_response_time,upstream_addr,upstream_status,connection,connection_requests,data.srcip,agent.ip,rule.mitre.tactic
id,,,,,,,,,,,,,,,,,,,,,
0,"Aug 29, 2023 @ 17:00:03.058",114.4.220.184,-,vm-director-nginx,30/Aug/2023:00:00:02 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,Mozilla/5.0 (Linux; Android 11; Redmi Note 8) ...,...,0.001,0.002,0.002,10.45.184.148:32442,404,688306444,2,114.4.220.184,192.168.1.99,"[""non-attack""]"
1,"Aug 29, 2023 @ 17:00:33.070",10.200.160.148,-,vm-director-nginx,30/Aug/2023:00:00:31 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,...,0,0.002,0.002,10.45.184.154:32442,404,688306865,2,10.200.160.148,192.168.1.99,"[""non-attack""]"
2,"Aug 29, 2023 @ 17:00:33.072",180.248.46.18,-,vm-director-nginx,30/Aug/2023:00:00:31 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...,...,0.001,0.002,0.002,10.45.184.148:32442,404,688306855,2,180.248.46.18,192.168.1.99,"[""non-attack""]"
3,"Aug 29, 2023 @ 17:00:33.076",125.166.3.1,-,vm-director-nginx,30/Aug/2023:00:00:32 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,Mozilla/5.0 (Linux; Android 12; M2003J15SC) Ap...,...,0,0.002,0.002,10.45.184.154:32442,404,688306869,2,125.166.3.1,192.168.1.99,"[""non-attack""]"
4,"Aug 29, 2023 @ 17:00:59.118",103.108.23.13,-,vm-director-nginx,30/Aug/2023:00:00:58 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...,...,0.001,0.003,0.003,10.45.184.154:32442,404,688307225,2,103.108.23.13,192.168.1.99,"[""non-attack""]"


In [141]:
ori_train_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 54480 entries, 0 to 54479
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   timestamp               54480 non-null  object 
 1   remote_addr             54480 non-null  object 
 2   remote_user             54480 non-null  object 
 3   hostname                54480 non-null  object 
 4   time_local              54480 non-null  object 
 5   request                 54480 non-null  object 
 6   status                  54480 non-null  int64  
 7   body_bytes_sent         54480 non-null  int64  
 8   http_referer            54480 non-null  object 
 9   http_user_agent         54480 non-null  object 
 10  http_x_forwarded_for    54480 non-null  object 
 11  time_iso8601            54480 non-null  object 
 12  request_method          54480 non-null  object 
 13  request_length          54480 non-null  int64  
 14  bytes_sent              54480 non-null

## Feature Engineering

In [142]:
ori_train_df.upstream_connect_time.value_counts()

0               27741
0.001           25571
0.002             526
-, 0.000           88
-                  87
-, -, -            70
0.004              66
0.003              64
0.005              49
0.008              36
0.006              33
0.007              20
0.009              14
0.012              12
0.01               11
0.018               9
0.026               6
0.02                5
0.013               5
0.016               5
0.017               4
0.011               4
0.014               4
0.021               4
-, 0.001            4
0.046               3
0.036               3
0.027               3
0.037               2
0.025               2
0.019               1
0.031               1
0.092               1
1.003               1
-, 0.013            1
-, 0.002            1
0.068               1
0.001, 0.001        1
0.032               1
0.034               1
0.067               1
0.489               1
0.04                1
0.035               1
0.265               1
0.169     

In [143]:
ori_train_df.upstream_response_time.value_counts()

0.002    24677
0.001    13095
0.003     6410
0         1310
0.004      691
         ...  
0.211        1
0.248        1
1.937        1
0.28         1
1.738        1
Name: upstream_response_time, Length: 874, dtype: int64

In [145]:
def clean_upstream_connect_time(x):
    x = x.split(', ')
    for i in range(len(x)):
        try:
            x[i] = float(x[i])
        except ValueError:
            x[i] = 0
    return np.max(x)
            

train_df = ori_train_df.copy()
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'], format='%b %d, %Y @ %H:%M:%S.%f')
train_df['upstream_connect_time'] = train_df['upstream_connect_time'].apply(clean_upstream_connect_time)
train_df['upstream_header_time'] = train_df['upstream_header_time'].apply(clean_upstream_connect_time)
train_df['upstream_response_time'] = train_df['upstream_response_time'].apply(clean_upstream_connect_time)
train_df.drop(train_df[train_df['rule.mitre.tactic'] == '["Credential Access"]'].index, inplace=True)
train_df.drop([
    'remote_addr',
    'remote_user',
    'hostname',
    'time_local',
    'request',
    'http_referer',
    'http_referer2',
    'http_x_forwarded_for',
    'http_user_agent',
    'time_iso8601',
    'upstream_addr',
    'upstream_status',
    # 'connection',
    'data.srcip',
    'agent.ip'
], axis=1, inplace=True)
cat_cols = ['status', 'request_method']

In [90]:
train_df.head()

,timestamp,status,body_bytes_sent,request_method,request_length,bytes_sent,request_time,upstream_connect_time,upstream_header_time,upstream_response_time,connection,connection_requests,rule.mitre.tactic
id,,,,,,,,,,,,,
0,2023-08-29 17:00:03.058,404,118,GET,79,330,0.002,0.001,0.002,0.002,688306444,2,"[""non-attack""]"
1,2023-08-29 17:00:33.070,404,118,GET,79,330,0.002,0.000,0.002,0.002,688306865,2,"[""non-attack""]"
2,2023-08-29 17:00:33.072,404,118,GET,92,330,0.003,0.001,0.002,0.002,688306855,2,"[""non-attack""]"
3,2023-08-29 17:00:33.076,404,118,GET,81,330,0.002,0.000,0.002,0.002,688306869,2,"[""non-attack""]"
4,2023-08-29 17:00:59.118,404,118,GET,82,330,0.003,0.001,0.003,0.003,688307225,2,"[""non-attack""]"


In [107]:
train_df = pd.get_dummies(train_df, columns=cat_cols)
train_df.head()

,timestamp,body_bytes_sent,request_length,bytes_sent,request_time,upstream_connect_time,upstream_header_time,upstream_response_time,connection,connection_requests,...,status_405,status_407,status_413,status_422,status_500,status_502,status_504,request_method_GET,request_method_HEAD,request_method_POST
id,,,,,,,,,,,,,,,,,,,,,
0,2023-08-29 17:00:03.058,118,79,330,0.002,0.001,0.002,0.002,688306444,2,...,0,0,0,0,0,0,0,1,0,0
1,2023-08-29 17:00:33.070,118,79,330,0.002,0.000,0.002,0.002,688306865,2,...,0,0,0,0,0,0,0,1,0,0
2,2023-08-29 17:00:33.072,118,92,330,0.003,0.001,0.002,0.002,688306855,2,...,0,0,0,0,0,0,0,1,0,0
3,2023-08-29 17:00:33.076,118,81,330,0.002,0.000,0.002,0.002,688306869,2,...,0,0,0,0,0,0,0,1,0,0
4,2023-08-29 17:00:59.118,118,82,330,0.003,0.001,0.003,0.003,688307225,2,...,0,0,0,0,0,0,0,1,0,0


In [91]:
train_df.upstream_connect_time.value_counts()

0.000    27986
0.001    25577
0.002      527
0.004       66
0.003       64
0.005       49
0.008       36
0.006       33
0.007       20
0.009       14
0.012       12
0.010       11
0.018        9
0.013        6
0.026        6
0.016        5
0.020        5
0.017        4
0.011        4
0.014        4
0.021        4
0.046        3
0.036        3
0.027        3
0.025        2
0.037        2
0.067        1
0.040        1
0.019        1
0.034        1
0.031        1
0.092        1
0.068        1
0.032        1
0.169        1
1.003        1
0.489        1
0.265        1
0.035        1
0.108        1
0.060        1
0.045        1
0.053        1
0.033        1
0.015        1
0.118        1
0.029        1
0.126        1
0.024        1
0.061        1
0.064        1
Name: upstream_connect_time, dtype: int64

## Modeling

In [146]:
X = train_df.drop('rule.mitre.tactic', axis=1)
y = train_df['rule.mitre.tactic']

X_train, X_val, y_train, y_val = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=train_df['rule.mitre.tactic']
                                                    )
X_train.shape, y_train.shape, X_val.shape, y_val.shape

((43583, 12), (43583,), (10896, 12), (10896,))

In [147]:
cbc = cb.CatBoostClassifier(task_type="GPU",
                            cat_features=cat_cols,
                            verbose=1, 
                            random_seed=42)
cbc.fit(X_train, y_train, eval_set=(X_val, y_val))

Learning rate set to 0.153958
0:	learn: 1.0684017	test: 1.0683893	best: 1.0683893 (0)	total: 35.9ms	remaining: 35.9s
1:	learn: 0.8124862	test: 0.8125243	best: 0.8125243 (1)	total: 61.9ms	remaining: 30.9s
2:	learn: 0.6468192	test: 0.6470068	best: 0.6470068 (2)	total: 87.4ms	remaining: 29s
3:	learn: 0.5285707	test: 0.5289176	best: 0.5289176 (3)	total: 111ms	remaining: 27.7s
4:	learn: 0.4395073	test: 0.4399308	best: 0.4399308 (4)	total: 137ms	remaining: 27.3s
5:	learn: 0.3706245	test: 0.3710709	best: 0.3710709 (5)	total: 163ms	remaining: 27s
6:	learn: 0.3160407	test: 0.3165519	best: 0.3165519 (6)	total: 188ms	remaining: 26.6s
7:	learn: 0.2718270	test: 0.2724629	best: 0.2724629 (7)	total: 210ms	remaining: 26.1s
8:	learn: 0.2360133	test: 0.2366943	best: 0.2366943 (8)	total: 234ms	remaining: 25.8s
9:	learn: 0.2066436	test: 0.2073824	best: 0.2073824 (9)	total: 251ms	remaining: 24.9s
10:	learn: 0.1823069	test: 0.1831026	best: 0.1831026 (10)	total: 260ms	remaining: 23.3s
11:	learn: 0.1618759	te

In [76]:
y_val_pred = cbc.predict(X_val)
print(classification_report(y_val, y_val_pred))

                     precision    recall  f1-score   support

["Defense Evasion"]       0.00      0.00      0.00         6
         ["Impact"]       1.00      1.00      1.00       357
 ["Initial Access"]       0.00      0.00      0.00         1
 ["Reconnaissance"]       0.00      0.00      0.00       112
     ["non-attack"]       0.99      1.00      0.99     10420

           accuracy                           0.99     10896
          macro avg       0.40      0.40      0.40     10896
       weighted avg       0.98      0.99      0.98     10896


d:\miniconda3\envs\tf\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
d:\miniconda3\envs\tf\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
d:\miniconda3\envs\tf\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [95]:
TARGET_MAP = {
    '["non-attack"]': 0,
    '["Defense Evasion"]': 1,
    '["Impact"]': 2,
    '["Initial Access"]': 3,
    '["Reconnaissance"]': 4,
    # '["Credential Access"]': 5,
}

In [135]:
counts = np.bincount(y.map(TARGET_MAP))
print(
    "Number of non-attack samples in training data: {} ({:.2f}% of total)".format(
        counts[0], 100 * float(counts[0]) / len(y)
    )
)

counts

Number of non-attack samples in training data: 52101 (95.64% of total)


array([52101,    29,  1784,     4,   561], dtype=int64)

In [136]:
weight_for_0 = 1.0 / counts[0]
weight_for_1 = 1.0 / counts[1]
weight_for_2 = 1.0 / counts[2]
weight_for_3 = 1.0 / counts[3]
weight_for_4 = 1.0 / counts[4]

In [137]:
from tensorflow import keras
from keras.utils import to_categorical

model = keras.Sequential(
    [
        keras.layers.Dense(
            256, activation="relu", input_shape=(X_train.drop('timestamp', axis=1).shape[-1],)
        ),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(5, activation="softmax"),
    ]
)
model.summary()

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_24 (Dense)            (None, 256)               6912      
                                                                 
 dense_25 (Dense)            (None, 256)               65792     
                                                                 
 dropout_12 (Dropout)        (None, 256)               0         
                                                                 
 dense_26 (Dense)            (None, 256)               65792     
                                                                 
 dropout_13 (Dropout)        (None, 256)               0         
                                                                 
 dense_27 (Dense)            (None, 5)                 1285      
                                                                 
Total params: 139,781
Trainable params: 139,781
Non-tr

In [138]:
def f1_score(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    recall = true_positives / (possible_positives + K.epsilon())
    return 2 * (precision * recall) / (precision + recall + K.epsilon())

metrics = [
    keras.metrics.FalseNegatives(name="fn"),
    keras.metrics.FalsePositives(name="fp"),
    keras.metrics.TrueNegatives(name="tn"),
    keras.metrics.TruePositives(name="tp"),
    keras.metrics.Precision(name="precision"),
    keras.metrics.Recall(name="recall"),
    f1_score,
]

model.compile(
    optimizer=keras.optimizers.Adam(1e-2), loss="categorical_crossentropy", metrics=metrics
)

callbacks = [keras.callbacks.ModelCheckpoint("fraud_model_at_epoch_{epoch}.h5")]
class_weight = {0: weight_for_0, 1: weight_for_1, 2: weight_for_2, 3: weight_for_3, 4: weight_for_4}

model.fit(
    X_train.drop('timestamp', axis=1),
    to_categorical(y_train.map(TARGET_MAP)),
    batch_size=2048,
    epochs=30,
    verbose=2,
    callbacks=callbacks,
    validation_data=(X_val.drop('timestamp', axis=1), to_categorical(y_val.map(TARGET_MAP))),
    class_weight=class_weight,
)

Epoch 1/30
22/22 - 2s - loss: 8365.6982 - fn: 34518.0000 - fp: 34518.0000 - tn: 139814.0000 - tp: 9065.0000 - precision: 0.2080 - recall: 0.2080 - f1_score: 0.2036 - val_loss: 3473638.0000 - val_fn: 10784.0000 - val_fp: 10784.0000 - val_tn: 32800.0000 - val_tp: 112.0000 - val_precision: 0.0103 - val_recall: 0.0103 - val_f1_score: 0.0108 - 2s/epoch - 72ms/step
Epoch 2/30
22/22 - 0s - loss: 214.8378 - fn: 32721.0000 - fp: 32675.0000 - tn: 141657.0000 - tp: 10862.0000 - precision: 0.2495 - recall: 0.2492 - f1_score: 0.2494 - val_loss: 3533.8762 - val_fn: 10890.0000 - val_fp: 10890.0000 - val_tn: 32694.0000 - val_tp: 6.0000 - val_precision: 5.5066e-04 - val_recall: 5.5066e-04 - val_f1_score: 4.8826e-04 - 214ms/epoch - 10ms/step
Epoch 3/30
22/22 - 0s - loss: 42.5692 - fn: 37336.0000 - fp: 27665.0000 - tn: 146667.0000 - tp: 6247.0000 - precision: 0.1842 - recall: 0.1433 - f1_score: 0.1617 - val_loss: 1.6639 - val_fn: 10896.0000 - val_fp: 0.0000e+00 - val_tn: 43584.0000 - val_tp: 0.0000e+00 -

In [139]:
y_val_pred = model.predict(X_val.drop('timestamp', axis=1))
y_val_pred = np.argmax(y_val_pred, axis=1)
print(classification_report(y_val.map(TARGET_MAP), y_val_pred))

341/341 [==============================] - 0s 1ms/step
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     10420
           1       0.00      0.00      0.00         6
           2       0.00      0.00      0.00       357
           3       0.00      1.00      0.00         1
           4       0.00      0.00      0.00       112

    accuracy                           0.00     10896
   macro avg       0.00      0.20      0.00     10896
weighted avg       0.00      0.00      0.00     10896


d:\miniconda3\envs\tf\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
d:\miniconda3\envs\tf\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
d:\miniconda3\envs\tf\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
